# ESP32 CL-HAR Paper Results Analysis

This notebook analyzes pilot results for the ESP32 CL-HAR paper. The focus is resource feasibility and prediction shift under RAM-only continual learning on an ESP32-WROOM-32 with an MPU6050 IMU.

Scope and interpretation:

- The embedded pipeline uses a frozen MicroFlow-32 feature extractor, `OnlineLayer32`, and RAM-only `ReplayBuffer32`.
- FIFO and reservoir-per-class replay are compared under the same replay memory budget.
- The strongest real-device pilot is `Sitting` vs upstairs-like vertical hand motion.
- The second segment is upward hand motion near the host PC, constrained by the USB cable. It is not a real staircase benchmark.
- These results are pilot feasibility evidence, not final SOTA 6-class HAR accuracy.


## Imports And Paths

The notebook expects to be run from the repository root. If it is opened from `notebooks/`, the path setup below moves one level up automatically.

In [ ]:
from pathlib import Path
import csv
import json

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "logs").exists() and (ROOT.parent / "logs").exists():
    ROOT = ROOT.parent

FIG_DIR = ROOT / "results" / "figures"
TABLE_DIR = ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "sit_up_comparison": ROOT / "logs" / "parsed" / "pilot_sit_up" / "sit_up_comparison_2026-05-09.csv",
    "sit_up_segment_eval": ROOT / "logs" / "parsed" / "pilot_sit_up" / "sit_up_segment_eval_2026-05-09.csv",
    "pilot_2class_comparison": ROOT / "logs" / "parsed" / "pilot_2class" / "pilot_2class_comparison_2026-05-09.csv",
    "pilot_2class_segment_eval": ROOT / "logs" / "parsed" / "pilot_2class" / "pilot_2class_segment_eval_2026-05-09.csv",
    "phase5_markdown_table": ROOT / "results" / "tables" / "phase5_pilot_results_2026-05-09.md",
}

print(f"Repository root: {ROOT}")
print(f"Figure output:   {FIG_DIR}")
print(f"Table output:    {TABLE_DIR}")


## Load Parsed CSV Files

The `pilot_sit_up` files are required for the main analysis. The earlier standing-like pilot files are optional and loaded only if present.

In [ ]:
def load_csv(name: str, required: bool = False) -> pd.DataFrame | None:
    path = PATHS[name]
    if not path.exists():
        level = "ERROR" if required else "WARN"
        print(f"[{level}] Missing {name}: {path}")
        if required:
            raise FileNotFoundError(path)
        return None
    print(f"Loaded {name}: {path}")
    return pd.read_csv(path)

sit_up_comparison = load_csv("sit_up_comparison", required=True)
sit_up_segment_eval = load_csv("sit_up_segment_eval", required=True)

pilot_2class_comparison = load_csv("pilot_2class_comparison", required=False)
pilot_2class_segment_eval = load_csv("pilot_2class_segment_eval", required=False)

if PATHS["phase5_markdown_table"].exists():
    print(f"Optional markdown table present: {PATHS['phase5_markdown_table']}")


## Raw Parsed Tables

The next cells display the already-parsed pilot CSVs without additional filtering.

In [ ]:
sit_up_comparison

In [ ]:
sit_up_segment_eval

## Clean And Format Metrics

The parsed logs store timing in microseconds and replay memory in bytes. For paper tables, inference is easier to read in milliseconds and replay memory in KiB.

In [ ]:
MODE_ORDER = ["no_adapt", "fifo", "reservoir"]

def safe_float(value, default=float("nan")):
    try:
        if pd.isna(value) or value == "":
            return default
        return float(value)
    except (TypeError, ValueError):
        return default

def us_to_ms(value):
    value = safe_float(value)
    return value / 1000.0

def bytes_to_kib(value):
    value = safe_float(value)
    return value / 1024.0

def format_percent(value):
    value = safe_float(value)
    if pd.isna(value):
        return "-"
    return f"{value:.3f}%"

def prepare_comparison(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    numeric_cols = [
        "pred_rows",
        "label_rows",
        "train_rows",
        "replay_ram_est",
        "pred_infer_us_mean",
        "pred_head_us_mean",
        "train_update_us_mean",
        "cl_update_vs_infer_pct",
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out["inference_ms"] = out["pred_infer_us_mean"] / 1000.0
    out["train_update_ms"] = out["train_update_us_mean"] / 1000.0
    out["replay_ram_kib"] = out["replay_ram_est"] / 1024.0

    if "cl_update_vs_infer_pct" not in out.columns or out["cl_update_vs_infer_pct"].isna().all():
        out["cl_update_vs_infer_pct"] = (
            out["train_update_us_mean"] / out["pred_infer_us_mean"] * 100.0
        )
    else:
        missing = out["cl_update_vs_infer_pct"].isna()
        out.loc[missing, "cl_update_vs_infer_pct"] = (
            out.loc[missing, "train_update_us_mean"]
            / out.loc[missing, "pred_infer_us_mean"]
            * 100.0
        )

    out["mode"] = pd.Categorical(out["mode"], categories=MODE_ORDER, ordered=True)
    return out.sort_values("mode").reset_index(drop=True)

sit_up_metrics = prepare_comparison(sit_up_comparison)
sit_up_metrics

## Table 1: Resource And CL Overhead

This table is the main resource result for the paper. It shows that the online CL update cost is below 1% of MicroFlow-32 inference latency in this pilot.

In [ ]:
resource_cols = [
    "mode",
    "pred_rows",
    "label_rows",
    "train_rows",
    "replay_ram_kib",
    "inference_ms",
    "pred_head_us_mean",
    "train_update_us_mean",
    "cl_update_vs_infer_pct",
]

table_resource = sit_up_metrics[resource_cols].copy()
table_resource = table_resource.rename(
    columns={
        "pred_head_us_mean": "head_us_mean",
    }
)

table_resource_path = TABLE_DIR / "table_resource_overhead_sit_up.csv"
table_resource_md_path = TABLE_DIR / "table_resource_overhead_sit_up.md"
table_resource.to_csv(table_resource_path, index=False)

def dataframe_to_markdown(df: pd.DataFrame) -> str:
    headers = list(df.columns)
    rows = []
    for _, row in df.iterrows():
        rows.append(["" if pd.isna(row[col]) else str(row[col]) for col in headers])
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in rows:
        escaped = [cell.replace("|", "\\|") for cell in row]
        lines.append("| " + " | ".join(escaped) + " |")
    return "\n".join(lines) + "\n"

table_resource_md_path.write_text(dataframe_to_markdown(table_resource), encoding="utf-8")

print(f"Saved: {table_resource_path}")
print(f"Saved: {table_resource_md_path}")
table_resource

## Plot Helpers

In [ ]:
def save_current_figure(stem: str):
    png_path = FIG_DIR / f"{stem}.png"
    pdf_path = FIG_DIR / f"{stem}.pdf"
    plt.tight_layout()
    plt.savefig(png_path, dpi=200)
    plt.savefig(pdf_path)
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")

def ordered_modes(series):
    return [mode for mode in MODE_ORDER if mode in set(series.astype(str))]


## Plot 1: MicroFlow-32 Inference Latency By Mode

In [ ]:
plot_df = sit_up_metrics.set_index("mode").loc[ordered_modes(sit_up_metrics["mode"])]

plt.figure(figsize=(6, 4))
plt.bar(plot_df.index.astype(str), plot_df["inference_ms"])
plt.title("MicroFlow-32 inference latency on ESP32")
plt.xlabel("Mode")
plt.ylabel("Inference time, ms")
save_current_figure("fig_inference_latency_sit_up")
plt.show()

## Plot 2: RAM-Only CL Update Cost

In [ ]:
cl_df = sit_up_metrics[sit_up_metrics["mode"].astype(str).isin(["fifo", "reservoir"])].copy()
cl_df = cl_df.set_index("mode").loc[["fifo", "reservoir"]]

plt.figure(figsize=(5, 4))
plt.bar(cl_df.index.astype(str), cl_df["train_update_us_mean"])
plt.title("RAM-only CL update cost")
plt.xlabel("Mode")
plt.ylabel("Update time, us")
save_current_figure("fig_cl_update_cost_sit_up")
plt.show()

## Plot 3: Accepted Rate By Segment And Mode

In [ ]:
segment_df = sit_up_segment_eval.copy()
segment_df["accepted_rate"] = pd.to_numeric(segment_df["accepted_rate"], errors="coerce")
segment_pivot = segment_df.pivot(index="mode", columns="segment", values="accepted_rate")
segment_pivot = segment_pivot.reindex([mode for mode in MODE_ORDER if mode in segment_pivot.index])

ax = segment_pivot.plot(kind="bar", figsize=(7, 4))
ax.set_title("Segment-level prediction agreement in real-device pilot")
ax.set_xlabel("Mode")
ax.set_ylabel("Accepted prediction rate")
ax.set_ylim(0, 1)
ax.legend(title="Segment")
save_current_figure("fig_segment_accepted_rate_sit_up")
plt.show()

## Plot 4: Prediction Shift On Upstairs-Like Vertical Motion

This is the key pilot figure: no adaptation stays near zero accepted agreement on the upstairs-like segment, while FIFO and reservoir shift predictions toward `Upstairs`/`Downstairs` after supervised labels.

In [ ]:
up_df = segment_df[segment_df["segment"] == "upstairs_like"].copy()
up_df["mode"] = pd.Categorical(up_df["mode"], categories=MODE_ORDER, ordered=True)
up_df = up_df.sort_values("mode")

plt.figure(figsize=(6, 4))
plt.bar(up_df["mode"].astype(str), up_df["accepted_rate"])
plt.title("Prediction shift on upstairs-like vertical motion")
plt.xlabel("Mode")
plt.ylabel("Accepted prediction rate")
plt.ylim(0, 1)
save_current_figure("fig_upstairs_like_shift_sit_up")
plt.show()

## Prediction Class Distribution Parsing

In [ ]:
def parse_pred_counts(raw: str) -> dict[str, int]:
    counts: dict[str, int] = {}
    if not isinstance(raw, str) or raw.strip() == "":
        return counts
    for part in raw.split(";"):
        if not part:
            continue
        label, value = part.split("=", maxsplit=1)
        counts[label] = int(value)
    return counts

dist_rows = []
for _, row in segment_df.iterrows():
    for predicted_label, count in parse_pred_counts(row["pred_counts"]).items():
        dist_rows.append(
            {
                "mode": row["mode"],
                "segment": row["segment"],
                "predicted_label": predicted_label,
                "count": count,
            }
        )

prediction_distribution = pd.DataFrame(dist_rows)
prediction_distribution_path = TABLE_DIR / "table_prediction_distribution_sit_up.csv"
prediction_distribution.to_csv(prediction_distribution_path, index=False)
print(f"Saved: {prediction_distribution_path}")
prediction_distribution

## Plot 5: Prediction Distribution For Upstairs-Like Segment

In [ ]:
up_dist = prediction_distribution[prediction_distribution["segment"] == "upstairs_like"].copy()
dist_pivot = up_dist.pivot_table(
    index="mode",
    columns="predicted_label",
    values="count",
    aggfunc="sum",
    fill_value=0,
)
dist_pivot = dist_pivot.reindex([mode for mode in MODE_ORDER if mode in dist_pivot.index])

ax = dist_pivot.plot(kind="bar", stacked=True, figsize=(7, 4))
ax.set_title("Predicted class distribution on upstairs-like segment")
ax.set_xlabel("Mode")
ax.set_ylabel("Prediction count")
ax.legend(title="Predicted label")
save_current_figure("fig_prediction_distribution_upstairs_like")
plt.show()

## Interpretation

The no-adaptation run classified the upstairs-like segment as `Sitting`, despite visible upward hand motion in the IMU stream and a clear drop in confidence. FIFO and reservoir adaptation shifted predictions toward the `Upstairs`/`Downstairs` classes after supervised UART labels. Reservoir showed a slightly higher accepted rate in this pilot, but this single short run is too small to claim statistical superiority.

The main publishable resource result is that the RAM-only CL update overhead is below 1% of MicroFlow-32 inference time. This supports the feasibility of replay-based in-session continual learning on ESP32. The pilot should be described as a real-device sanity check, not as a full 6-class HAR benchmark and not as a real staircase benchmark.

## Optional: Compare With Previous Standing-Like Pilot

The previous pilot is kept as a secondary sanity check. It should not be overemphasized because the second segment was standing-like small movement rather than a clean HAR activity.

In [ ]:
def compact_run_table(df: pd.DataFrame, pilot_name: str) -> pd.DataFrame:
    metrics = prepare_comparison(df)
    out = metrics[["mode", "inference_ms", "train_update_us_mean"]].copy()
    out.insert(0, "pilot_name", pilot_name)
    return out

optional_tables = [compact_run_table(sit_up_comparison, "sitting_vs_upstairs_like")]
if pilot_2class_comparison is not None:
    optional_tables.append(compact_run_table(pilot_2class_comparison, "earlier_standing_like"))

pilot_comparison_compact = pd.concat(optional_tables, ignore_index=True)
pilot_comparison_compact_path = TABLE_DIR / "table_optional_pilot_comparison.csv"
pilot_comparison_compact.to_csv(pilot_comparison_compact_path, index=False)
print(f"Saved: {pilot_comparison_compact_path}")
pilot_comparison_compact

## Final Paper-Ready Summary

- The ESP32-WROOM-32 prototype executed MicroFlow-32 inference at approximately 172 ms per window.
- The RAM-only `OnlineLayer32` update took approximately 0.66 ms for FIFO/reservoir replay in the `Sitting` vs upstairs-like vertical hand-motion pilot.
- Replay memory was fixed at `6 x 16 x 32 x 4` bytes, approximately 12 KiB.
- In the `Sitting` vs upstairs-like pilot, no adaptation remained at 0% accepted rate on the upstairs-like segment, while FIFO and reservoir reached approximately 88.6% and 93.9% respectively.
- These results support the feasibility of replay-based in-session continual learning on ESP32, with non-volatile persistence left for future work.
